# ML Tutor — Week 6: Tuning, fairness & explainability
**Track:** Core ML · **Lab:** Lab 6 — Tune, audit, and explain an equitable recommendation model

**Objectives this week:**
1. Tune models with grid/random search and read validation curves. *(CO5)*
2. Audit performance across patient subgroups and read SHAP/LIME explanations. *(CO6)*
3. Mitigate an observed fairness gap and communicate model drivers. *(CO5, CO6)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~25 min)
**Goal for this session:** Arrive able to say what leakage is, why cross-validation is safer than one lucky split, and why an accurate model can still perform unevenly across subgroups. Reference delta: use one tuned model plus one baseline; keep fairness and explainability extras in Advanced material.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. From Week 5, why do we keep preprocessing inside the validation loop instead of doing it once on the full dataset?
2. If two readmission models have the same accuracy but one saw future information during training, which one do you trust and why?
3. What is one reason a model that works overall might still be unfair for a particular patient subgroup?


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Synthetic treatment-support dataset with columns such as age_band, comorbidity_count, prior_utilization, adherence_score, access_barrier_flag, subgroup, and recommended_support (binary target). All records are fictional and for educational model-evaluation practice only.
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-06/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-06/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — Hyperparameter tuning
**Lane A · the general idea:** Hyperparameters such as max_depth, min_samples_leaf, or C are knobs chosen before fitting. A search procedure tries combinations and evaluates them inside cross-validation so we do not mistake noise for learning.

**Lane B · the same idea in pharmacy terms:** For a treatment-recommendation model, a deeper tree may fit the training cohort perfectly but behave erratically on new patients. Tuning is how we choose a setting that travels better, not how we win a leaderboard screenshot.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — one knob (max_depth), judged honestly with cross-validation
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification

# think: 120 patients, 5 features, target = responded to therapy (synthetic)
X, y = make_classification(n_samples=120, n_features=5, random_state=0)

for depth in [1, 3, 10]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    scores = cross_val_score(tree, X, y, cv=5)
    print(f"max_depth={depth:2d}  mean CV accuracy = {scores.mean():.2f}")
# the winner is chosen by CROSS-VALIDATION — the test set stays untouched.


## 1. Build a safe baseline pipeline
Create a train/test split and a pipeline with preprocessing plus a baseline decision tree.

**Checkpoint:** Checkpoint 1 verifies that the train/test split exists, preprocessing is inside the pipeline, and the baseline model is fit without touching the test labels during training.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

# df: synthetic treatment-support dataset (fictional, for evaluation practice only).
df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "comorbidity_count": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "prior_utilization": [1, 4, 8, 2, 0, 6, 3, 1, 9, 3, 0, 7, 2, 1, 6, 3],
    "adherence_score": [0.9, 0.6, 0.4, 0.8, 0.95, 0.5, 0.7, 0.85, 0.3, 0.75,
                         0.92, 0.45, 0.8, 0.9, 0.55, 0.7],
    "access_barrier_flag": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "subgroup": ["A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B"],
    "recommended_support": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

target = "recommended_support"
feature_cols = [
    "age_band",
    "comorbidity_count",
    "prior_utilization",
    "adherence_score",
    "access_barrier_flag",
    "subgroup",
]

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=7,
)

numeric_features = ["comorbidity_count", "prior_utilization", "adherence_score"]
categorical_features = ["age_band", "access_barrier_flag", "subgroup"]

# TODO: build a preprocessing transformer for numeric + categorical features
preprocess = ...

baseline = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(random_state=7)),
    ]
)

# TODO: fit baseline on the TRAIN split only



**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Tune one model honestly
Run a small grid search over tree depth and leaf size using recall as the scoring metric, then compare tuned versus baseline results.

**Checkpoint:** Checkpoint 2 verifies that GridSearchCV ran on the training split, used a valid scoring string, produced a best-parameter dictionary plus mean CV score, and included a short tuned-vs-baseline comparison note.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

# df: synthetic treatment-support dataset (fictional, for evaluation practice only).
df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "comorbidity_count": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "prior_utilization": [1, 4, 8, 2, 0, 6, 3, 1, 9, 3, 0, 7, 2, 1, 6, 3],
    "adherence_score": [0.9, 0.6, 0.4, 0.8, 0.95, 0.5, 0.7, 0.85, 0.3, 0.75,
                         0.92, 0.45, 0.8, 0.9, 0.55, 0.7],
    "access_barrier_flag": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "subgroup": ["A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B"],
    "recommended_support": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

target = "recommended_support"
feature_cols = ["age_band", "comorbidity_count", "prior_utilization",
                "adherence_score", "access_barrier_flag", "subgroup"]
X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7,
)

numeric_features = ["comorbidity_count", "prior_utilization", "adherence_score"]
categorical_features = ["age_band", "access_barrier_flag", "subgroup"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

baseline = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(random_state=7)),
    ]
)
baseline.fit(X_train, y_train)

param_grid = {
    "model__max_depth": [2, 4, 6, None],
    "model__min_samples_leaf": [1, 3, 6],
}

search = GridSearchCV(
    estimator=baseline,
    param_grid=param_grid,
    scoring=...,      # TODO: choose the metric we care about today
    cv=5,
    n_jobs=None,
)

# TODO: fit search on X_train, y_train

print("Best params:", ...)
print("Best mean CV score:", ...)

# TODO: compare the tuned mean CV recall against your untuned baseline.
# In one comment, write whether the improvement is meaningful or tiny.



**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Audit subgroup performance
Compare recall and false-negative counts across subgroups on held-out predictions, then decide whether the gap is operationally important.

**Checkpoint:** Checkpoint 3 verifies that students produced a table with subgroup, recall, false_negatives, and n, identified which subgroup had the lowest recall on the held-out set, and wrote a brief judgment note rather than only printing numbers.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score, confusion_matrix

# df: synthetic treatment-support dataset (fictional, for evaluation practice only).
df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "comorbidity_count": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "prior_utilization": [1, 4, 8, 2, 0, 6, 3, 1, 9, 3, 0, 7, 2, 1, 6, 3],
    "adherence_score": [0.9, 0.6, 0.4, 0.8, 0.95, 0.5, 0.7, 0.85, 0.3, 0.75,
                         0.92, 0.45, 0.8, 0.9, 0.55, 0.7],
    "access_barrier_flag": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "subgroup": ["A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B"],
    "recommended_support": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

target = "recommended_support"
feature_cols = ["age_band", "comorbidity_count", "prior_utilization",
                "adherence_score", "access_barrier_flag", "subgroup"]
X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7,
)

numeric_features = ["comorbidity_count", "prior_utilization", "adherence_score"]
categorical_features = ["age_band", "access_barrier_flag", "subgroup"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

baseline = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(random_state=7)),
    ]
)

param_grid = {
    "model__max_depth": [2, 4, 6, None],
    "model__min_samples_leaf": [1, 3, 6],
}
search = GridSearchCV(estimator=baseline, param_grid=param_grid, scoring="recall", cv=5, n_jobs=None)
search.fit(X_train, y_train)

best_model = ...   # TODO: recover the tuned estimator
y_pred = ...       # TODO: predict on X_test

audit = X_test[["subgroup"]].copy()
audit["y_true"] = y_test.to_numpy()
audit["y_pred"] = y_pred

rows = []
for subgroup_name, group_df in audit.groupby("subgroup"):
    # TODO: compute recall for this subgroup
    subgroup_recall = ...

    # TODO: count false negatives from y_true / y_pred
    false_negatives = ...

    rows.append(
        {
            "subgroup": subgroup_name,
            "recall": subgroup_recall,
            "false_negatives": false_negatives,
            "n": len(group_df),
        }
    )

audit_table = pd.DataFrame(rows).sort_values("recall")
print(audit_table)

# TODO: write 2-3 sentences:
# - Which subgroup had the weakest recall?
# - Is the gap large enough to worry you in this course scenario?
# - What extra evidence would you want before changing the model?



**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Add one explanation view and propose one mitigation
Generate a lightweight explanation artifact and write a short mitigation note grounded in the audit results.

**Checkpoint:** Checkpoint 4 verifies that the notebook produces at least one explanation artifact or ranked feature list and includes a short mitigation note tied to the subgroup audit, not just the global score.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

# df: synthetic treatment-support dataset (fictional, for evaluation practice only).
df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "comorbidity_count": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "prior_utilization": [1, 4, 8, 2, 0, 6, 3, 1, 9, 3, 0, 7, 2, 1, 6, 3],
    "adherence_score": [0.9, 0.6, 0.4, 0.8, 0.95, 0.5, 0.7, 0.85, 0.3, 0.75,
                         0.92, 0.45, 0.8, 0.9, 0.55, 0.7],
    "access_barrier_flag": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "subgroup": ["A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B", "A", "B"],
    "recommended_support": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

target = "recommended_support"
feature_cols = ["age_band", "comorbidity_count", "prior_utilization",
                "adherence_score", "access_barrier_flag", "subgroup"]
X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7,
)

numeric_features = ["comorbidity_count", "prior_utilization", "adherence_score"]
categorical_features = ["age_band", "access_barrier_flag", "subgroup"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

baseline = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(random_state=7)),
    ]
)

param_grid = {
    "model__max_depth": [2, 4, 6, None],
    "model__min_samples_leaf": [1, 3, 6],
}
search = GridSearchCV(estimator=baseline, param_grid=param_grid, scoring="recall", cv=5, n_jobs=None)
search.fit(X_train, y_train)
best_model = search.best_estimator_

# Choose ONE explanation path based on what is available in the notebook environment.
# Do not skip directly here: only explain the tuned model AFTER you have checked
# its validation behavior and subgroup audit.

explanation_note = []

# Option A: global permutation importance
from sklearn.inspection import permutation_importance

result = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=7,
)

# TODO: build top_features as the 5 highest-importance feature names (list of str)
top_features = ...
print("Top features:", top_features)

# Option B: if SHAP is available, explain a small sample
# import shap
# explainer = shap.Explainer(best_model.predict, X_sample_transformed)
# shap_values = explainer(X_sample_transformed)

# TODO: write 3-4 lines as a single string assigned to mitigation_note:
# 1. Which subgroup metric worried you most?
# 2. Which feature(s) seemed influential?
# 3. One mitigation you would test next (threshold, feature review, reweighting, or model change)
# 4. One reason this explanation should be treated cautiously
mitigation_note = ...



**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“Hyperparameter tuning means trying many settings until the test score looks best.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: HW6 — Fairness memo for a tuned model
Starting from your lab notebook, compare two candidate models or two threshold settings for the same recommendation task. Report overall recall plus at least two subgroup metrics, generate one explanation artifact, and write a brief memo recommending which version is safer to continue studying and why.

**Deliverable:** One runnable notebook plus a one-page memo. The memo must name the validation setup, the chosen metric, one observed subgroup gap, one limitation of the explanation method used, and one next mitigation to test. Do not give clinical treatment advice; stay at the model-evaluation level.


In [ ]:
# HW / challenge workspace — build it here

